In [ ]:
from typing import Tuple
import numpy as np

from ...util.coordinate import Coordinate
from ...util.levy_flight import levy_flight


# BASE ABELHA

In [ ]:
class BeeBase(Coordinate):

    def __init__(self, **kwargs) -> None:
        """
        Initializes a new instance of the Bee class
        """
        super().__init__(**kwargs)
        self.__limit = kwargs.get('trials', 3)
        self.__lambda = kwargs.get('lambda', 1.5)
        self.__alpha = kwargs.get('alpha', 1.)
        self.__trials = 0
        self.__reset = True

    @property
    def is_reset(self) -> bool:
        """
        Indicates whether the bee is reset or not.

        Returns:
            bool: True if the bee was reset otherwise False
        """
        return self.__reset

    def reset(self) -> None:
        """
        Reset the bee if it exceeded the trial limit.
        """
        if self.__trials >= self.__limit:
            self._initialize()
            self.__trials = 0
            self.__reset = True

    def _explore(self, starting_position: Tuple[float, float], start_value: float) -> None:
        """
        Try to generate a new, position and save the better one

        Args:
            starting_position (Tuple[float, float]): The starting position
            start_value (float): The positions value
        """
        new_pos = levy_flight(starting_position, self.__alpha, self.__lambda, self._random)
        new_value = self._function(new_pos)

        if new_value < start_value:
            self._position = new_pos
            self.__trials = 0
            self.__reset = False
        else:
            self.__trials += 1

# ABELHA OPERÁRIA / EXPLORADORA

In [ ]:
class EmployeeBee(BeeBase):
    def explore(self) -> None:
        """
        Explore new food sources from it's own position
        """
        self._explore(self._position, self.value)

    @property
    def fitness(self) -> float:
        """
        Get the fitness. Used for probability calculations

        Returns:
            float: the fitness
        """

        # Prefer negative values
        if self.value > 0:
            fitness = 1 / (self.value+1)  # shift the value by a constant
        else:
            fitness = np.abs(self.value) + 1

        return fitness

# ABELHA OBSERVADORA

In [ ]:
class OnlookerBee(BeeBase):
    def explore(self, starting_position: Tuple[float, float], start_value: float) -> None:
        """
        Explore new food sources from the given one

        Args:
            starting_position ([type]): [description]
            start_value (float): [description]
        """
        self._explore(starting_position, start_value)

# ABC PROBLEM

In [ ]:
from copy import deepcopy
from functools import reduce
import logging

from ..util.problem_base import ProblemBase

LOGGER = logging.getLogger(__name__)

class ABCProblem(ProblemBase):
    """Artificial Bee Colony Problem"""

    def __init__(self, **kwargs):
        """
        Initializes a new instance of the ABCProblem class.
        """
        super().__init__(**kwargs)
        self.__iteration_number = kwargs['iteration_number']
        self.__employee_bees = [
            EmployeeBee(**kwargs, bit_generator=self._random)
            for _ in range(kwargs['bees'])
        ]

        self.__onlooker_bees = [
            OnlookerBee(**kwargs, bit_generator=self._random)
            for _ in range(kwargs['bees'])
        ]

        self._visualizer = Visualizer(**kwargs)

    def solve(self):
        """
        Solve the ABC problem
        """
        best = min(self.__employee_bees + self.__onlooker_bees, key=lambda bee: bee.value)
        self._visualizer.add_data(employee_bees=self.__employee_bees, onlooker_bees=self.__onlooker_bees, best_position=best.position)

        for iteration in range(self.__iteration_number):
            # Employee bee phase
            for bee in self.__employee_bees:
                bee.explore()

            # Calculate the employee bees fitness values and probabilities
            overall_fitness = reduce(lambda acc, curr: acc + curr.fitness, self.__employee_bees, 0)
            employee_bees_fitness_probs = [bee.fitness/overall_fitness for bee in self.__employee_bees]

            # Choose the employee bees positions proportional to their fitness
            choices = self._random.choice(self.__employee_bees, size=len(self.__employee_bees), p=employee_bees_fitness_probs)

            # Onlooker phase
            # Explore new food sources based on the chosen employees' food sources
            for bee, choice in zip(self.__onlooker_bees, choices):
                bee.explore(choice.position, choice.value)

            # Scout phase
            for bee in self.__employee_bees + self.__onlooker_bees:
                bee.reset()

             # Update best food source
            current_best = min(self.__employee_bees + self.__onlooker_bees)
            if current_best < best:
                best = deepcopy(current_best)
                LOGGER.info('Iteration %i Found new best solution="%s" at position="%s"', iteration+1, best.value, best.position)

            # Add data for plotting
            self._visualizer.add_data(employee_bees=self.__employee_bees, onlooker_bees=self.__onlooker_bees, best_position=best.position)

        return best